# 03 — Position Reconstruction

**Goals**
- Q5: can we reconstruct position from acceleration? Challenges?
- Q6: implement reconstruction; show 2020-12-25 02:01–02:11; estimate stroke amplitude

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from well_analysis.data import load_test_data
from well_analysis.signal import check_even_sampling
from well_analysis.signal.integration import integrate_acceleration, estimate_gravity_offset
from well_analysis.detection import detect_well_state
from well_analysis.analysis.dynamometer import estimate_stroke_amplitude
from well_analysis.viz import plot_time_series

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

df = load_test_data()
_, fs = check_even_sampling(df['Timestamp'])
accel = df['Acceleration'].values
is_running = detect_well_state(accel, fs=fs)

## 1. Can we reconstruct position? Challenges (Q5)

Position = ∬ a(t) dt².

**Challenges**:
1. **Gravity offset**: raw acceleration includes −g + sensor bias → must be removed before integrating.
2. **Integration drift**: each integration of numerical noise adds a ramp (linear drift after 1st, quadratic after 2nd). Even tiny DC errors blow up.
3. **Initial conditions**: velocity and position at t=0 are unknown.

**Solution**: high-pass filter (cutoff ≪ pump freq ≈ 0.05 Hz, e.g. 0.02 Hz) applied after each integration step. This removes accumulated drift while preserving the oscillatory signal of interest.

In [ ]:
# Estimate gravity offset from stopped intervals
g_offset = estimate_gravity_offset(accel, is_running)
print(f"Gravity offset estimate: {g_offset:.4f} m/s²  (expected ≈ -9.81)")

velocity, position = integrate_acceleration(accel, fs=fs, gravity_offset=g_offset, hp_cutoff=0.02)

## 2. Show reconstruction 2020-12-25 02:01 → 02:11 (Q6)

In [ ]:
t_start = pd.Timestamp('2020-12-25T02:01:00', tz='UTC')
t_end   = pd.Timestamp('2020-12-25T02:11:00', tz='UTC')
mask = (df['Timestamp'] >= t_start) & (df['Timestamp'] < t_end)

ts_sub  = df.loc[mask, 'Timestamp']
acc_sub = accel[mask]
pos_sub = position[mask]

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
plot_time_series(ts_sub, acc_sub, ylabel='Acceleration (m/s²)',
                 title='Acceleration 2020-12-25 02:01–02:11', ax=axes[0])
plot_time_series(ts_sub, pos_sub, ylabel='Position (m)',
                 title='Reconstructed Position', ax=axes[1])
plt.tight_layout()
plt.show()

In [ ]:
stroke = estimate_stroke_amplitude(pos_sub)
print(f"Estimated stroke amplitude: {stroke:.3f} m  ({stroke*100:.1f} cm)")